In [2]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from scapy.utils import PcapReader
from scapy.layers.inet import IP, TCP, UDP, ICMP
from scapy.layers.l2 import ARP

In [12]:
DATASET_PATH = Path("/home/emanuele/Documentos/ProjetosFaculdade/ProjetoIoT/tagged-2021")

pcaps = [
    p for p in DATASET_PATH.rglob("*.pcap")
    if not p.name.startswith("._")
]

print(f"Arquivos encontrados: {len(pcaps)}")

Arquivos encontrados: 7127


In [13]:
def extrair_features(caminho):
    total_packets = 0
    total_bytes = 0

    tcp = udp = icmp = arp = dns = 0
    tamanhos = []

    ips_origem = set()
    ips_destino = set()
    portas = set()

    primeiro_tempo = None
    ultimo_tempo = None

    with PcapReader(str(caminho)) as captura:
        for p in captura:
            total_packets += 1

            tamanho = len(p)
            total_bytes += tamanho
            tamanhos.append(tamanho)

            tempo = float(p.time)
            if primeiro_tempo is None:
                primeiro_tempo = tempo
            ultimo_tempo = tempo

            if IP in p:
                ips_origem.add(p[IP].src)
                ips_destino.add(p[IP].dst)

            if TCP in p:
                tcp += 1
                portas.add(p[TCP].dport)

            if UDP in p:
                udp += 1
                portas.add(p[UDP].dport)

                if p[UDP].sport == 53 or p[UDP].dport == 53:
                    dns += 1

            if ICMP in p:
                icmp += 1

            if ARP in p:
                arp += 1

    duracao = 0
    if primeiro_tempo is not None and ultimo_tempo is not None:
        duracao = ultimo_tempo - primeiro_tempo

    return {
        "device": caminho.parents[1].name,
        "arquivo": caminho.name,
        "total_packets": total_packets,
        "total_bytes": total_bytes,
        "avg_packet_size": total_bytes / total_packets if total_packets else 0,
        "max_packet_size": max(tamanhos) if tamanhos else 0,
        "min_packet_size": min(tamanhos) if tamanhos else 0,
        "tcp_packets": tcp,
        "udp_packets": udp,
        "dns_packets": dns,
        "icmp_packets": icmp,
        "arp_packets": arp,
        "unique_source_ips": len(ips_origem),
        "unique_destination_ips": len(ips_destino),
        "unique_ports": len(portas),
        "capture_duration": duracao,
        "packets_per_second": total_packets / duracao if duracao > 0 else 0,
        "bytes_per_second": total_bytes / duracao if duracao > 0 else 0,
        "tcp_ratio": tcp / total_packets if total_packets else 0,
        "udp_ratio": udp / total_packets if total_packets else 0,
        "dns_ratio": dns / total_packets if total_packets else 0,
        "icmp_ratio": icmp / total_packets if total_packets else 0,
    }

In [14]:
teste = extrair_features(pcaps[0])
teste

{'device': 'google-home-mini',
 'arquivo': '2021-08-04_04:25:06.19s.pcap',
 'total_packets': 253,
 'total_bytes': 172866,
 'avg_packet_size': 683.2648221343874,
 'max_packet_size': 1392,
 'min_packet_size': 66,
 'tcp_packets': 3,
 'udp_packets': 250,
 'dns_packets': 4,
 'icmp_packets': 0,
 'arp_packets': 0,
 'unique_source_ips': 5,
 'unique_destination_ips': 5,
 'unique_ports': 8,
 'capture_duration': 16.220131158828735,
 'packets_per_second': 15.597900998617403,
 'bytes_per_second': 10657.497051490103,
 'tcp_ratio': 0.011857707509881422,
 'udp_ratio': 0.9881422924901185,
 'dns_ratio': 0.015810276679841896,
 'icmp_ratio': 0.0}

In [15]:
import time

inicio = time.time()
teste = extrair_features(pcaps[0])
fim = time.time()

print(teste)
print(f"Tempo: {fim - inicio:.2f} segundos")

{'device': 'google-home-mini', 'arquivo': '2021-08-04_04:25:06.19s.pcap', 'total_packets': 253, 'total_bytes': 172866, 'avg_packet_size': 683.2648221343874, 'max_packet_size': 1392, 'min_packet_size': 66, 'tcp_packets': 3, 'udp_packets': 250, 'dns_packets': 4, 'icmp_packets': 0, 'arp_packets': 0, 'unique_source_ips': 5, 'unique_destination_ips': 5, 'unique_ports': 8, 'capture_duration': 16.220131158828735, 'packets_per_second': 15.597900998617403, 'bytes_per_second': 10657.497051490103, 'tcp_ratio': 0.011857707509881422, 'udp_ratio': 0.9881422924901185, 'dns_ratio': 0.015810276679841896, 'icmp_ratio': 0.0}
Tempo: 0.04 segundos


In [7]:
from tqdm import tqdm

dataset = []

for pcap in tqdm(pcaps):
    try:
        dataset.append(extrair_features(pcap))
    except Exception as e:
        print(f"Erro em {pcap}: {e}")

df = pd.DataFrame(dataset)

df.head()

100%|██████████| 7127/7127 [30:10<00:00,  3.94it/s]  


,device,arquivo,total_packets,total_bytes,avg_packet_size,max_packet_size,min_packet_size,tcp_packets,udp_packets,dns_packets,...,unique_source_ips,unique_destination_ips,unique_ports,capture_duration,packets_per_second,bytes_per_second,tcp_ratio,udp_ratio,dns_ratio,icmp_ratio
0,google-home-mini,2021-08-04_04:25:06.19s.pcap,253,172866,683.264822,1392,66,3,250,4,...,5,5,8,16.220131,15.597901,10657.497051,0.011858,0.988142,0.015810,0.000000
1,google-home-mini,2021-08-04_01:26:42.19s.pcap,325,210396,647.372308,1392,66,11,306,4,...,7,7,9,17.513915,18.556673,12013.076417,0.033846,0.941538,0.012308,0.024615
2,google-home-mini,2021-08-04_04:25:46.19s.pcap,273,201745,738.992674,1392,42,7,264,0,...,5,6,8,16.562513,16.483006,12180.820720,0.025641,0.967033,0.000000,0.000000
3,google-home-mini,2021-08-04_03:49:21.19s.pcap,290,175417,604.886207,1484,42,20,260,8,...,6,6,10,16.240346,17.856762,10801.309312,0.068966,0.896552,0.027586,0.027586
4,google-home-mini,2021-08-04_03:13:36.19s.pcap,277,175957,635.223827,1484,42,20,255,4,...,6,6,11,10.158871,27.266810,17320.527165,0.072202,0.920578,0.014440,0.000000


In [8]:
df.to_csv("csv_idle/dataset_iot.csv", index=False)

print("CSV salvo com sucesso!")
print(df.shape)

CSV salvo com sucesso!
(7127, 22)


In [9]:
df.shape

(7127, 22)

In [10]:
df.head()

,device,arquivo,total_packets,total_bytes,avg_packet_size,max_packet_size,min_packet_size,tcp_packets,udp_packets,dns_packets,...,unique_source_ips,unique_destination_ips,unique_ports,capture_duration,packets_per_second,bytes_per_second,tcp_ratio,udp_ratio,dns_ratio,icmp_ratio
0,google-home-mini,2021-08-04_04:25:06.19s.pcap,253,172866,683.264822,1392,66,3,250,4,...,5,5,8,16.220131,15.597901,10657.497051,0.011858,0.988142,0.015810,0.000000
1,google-home-mini,2021-08-04_01:26:42.19s.pcap,325,210396,647.372308,1392,66,11,306,4,...,7,7,9,17.513915,18.556673,12013.076417,0.033846,0.941538,0.012308,0.024615
2,google-home-mini,2021-08-04_04:25:46.19s.pcap,273,201745,738.992674,1392,42,7,264,0,...,5,6,8,16.562513,16.483006,12180.820720,0.025641,0.967033,0.000000,0.000000
3,google-home-mini,2021-08-04_03:49:21.19s.pcap,290,175417,604.886207,1484,42,20,260,8,...,6,6,10,16.240346,17.856762,10801.309312,0.068966,0.896552,0.027586,0.027586
4,google-home-mini,2021-08-04_03:13:36.19s.pcap,277,175957,635.223827,1484,42,20,255,4,...,6,6,11,10.158871,27.266810,17320.527165,0.072202,0.920578,0.014440,0.000000


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7127 entries, 0 to 7126
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   device                  7127 non-null   object 
 1   arquivo                 7127 non-null   object 
 2   total_packets           7127 non-null   int64  
 3   total_bytes             7127 non-null   int64  
 4   avg_packet_size         7127 non-null   float64
 5   max_packet_size         7127 non-null   int64  
 6   min_packet_size         7127 non-null   int64  
 7   tcp_packets             7127 non-null   int64  
 8   udp_packets             7127 non-null   int64  
 9   dns_packets             7127 non-null   int64  
 10  icmp_packets            7127 non-null   int64  
 11  arp_packets             7127 non-null   int64  
 12  unique_source_ips       7127 non-null   int64  
 13  unique_destination_ips  7127 non-null   int64  
 14  unique_ports            7127 non-null   